In [11]:
# pip install typing_extensions --upgrade
# pip install albumentations --upgrade

In [12]:
import os
import shutil
import cv2
import albumentations as A
import numpy as np
import random

# =====================================================
# CLASS CONFIG
# =====================================================
CLASS_MAP = {
    0: "Abnormal mononuclear cell",
    1: "Atypical-lymphocyte",
    2: "Basophil",
    3: "Cells-stained-dark-blue-with-fused-boundaries",
    4: "Eosinophil",
    5: "Lymphocyte",
    6: "Macrophage",
    7: "Mesothelial cell",
    8: "Mitotic cell",
    9: "Monocyte",
    10: "Neutrophil",
    11: "Plasma-cell",
    12: "Signet-ring cell"
}

# =====================================================
# USER CONFIG
# =====================================================
TARGET_CLASS_ID = 4
TARGET_CLASS_NAME = CLASS_MAP[TARGET_CLASS_ID]

INPUT_IMAGE_DIR  = f"dataset_no-aug/Raw/valid/{TARGET_CLASS_NAME}"
OUTPUT_IMAGE_DIR = f"dataset_aug/Raw/valid/{TARGET_CLASS_NAME}"

MAX_CLASS_REFERENCE = 1150

BG_COLOR_BGR = (196, 208, 188)
BG_COLOR_RGB = BG_COLOR_BGR[::-1]

# =====================================================
# MAIN FUNCTION
# =====================================================
def augment_then_copy_originals(
    input_dir: str,
    output_dir: str,
    max_reference: int,
    bg_color_rgb: tuple
):
    os.makedirs(output_dir, exist_ok=True)

    # -------------------------------------------------
    # 1) LOAD ORIGINAL FILES
    # -------------------------------------------------
    original_files = [
        f for f in os.listdir(input_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))
    ]

    num_originals = len(original_files)
    if num_originals == 0:
        print("❌ No images found in input directory.")
        return

    # -------------------------------------------------
    # 2) CALCULATE EXACT QUOTA PER IMAGE  ⭐ เลขเป๊ะ
    # -------------------------------------------------
    augment_needed = max_reference - num_originals

    print("===================================")
    print(f"Class               : {TARGET_CLASS_NAME}")
    print(f"Original images     : {num_originals}")
    print(f"Target Reference    : {max_reference}")

    if augment_needed <= 0:
        print("✅ No new images needed. Target already reached.")
        quota_list = [0] * num_originals
    else:
        base_augments = augment_needed // num_originals
        remainder     = augment_needed % num_originals  # ⭐ เศษที่เหลือ (เลขนี้ต้องแจกให้ครบ ไม่สุ่ม)

        # ⭐ สร้าง list โควตา: remainder ภาพแรกได้ base+1, ที่เหลือได้ base
        # แล้ว shuffle เพื่อกระจายว่าภาพไหนจะได้โบนัส (ยอดรวมเท่าเดิมเสมอ)
        quota_list = [base_augments + 1] * remainder + [base_augments] * (num_originals - remainder)
        random.shuffle(quota_list)

        print(f"Base augments/image : {base_augments} times")
        print(f"Images getting +1   : {remainder} images")
        print(f"Exact total to gen  : {augment_needed} images")
    print("===================================")

    # -------------------------------------------------
    # 3) AUGMENT PIPELINE
    # -------------------------------------------------
    augment_pipeline = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(
            limit=180,
            border_mode=cv2.BORDER_CONSTANT,
            fill=bg_color_rgb,
            p=1.0
        ),
        A.OneOf([
            A.ElasticTransform(
                alpha=35, # จากเดิม 60
                sigma=10,
                border_mode=cv2.BORDER_CONSTANT,
                fill=bg_color_rgb,
                p=1.0
            ),
        A.GridDistortion(
            num_steps=5,
            distort_limit=0.20,
            border_mode=cv2.BORDER_CONSTANT,
            fill=bg_color_rgb,
            p=1.0
        )
    ], p=0.70),
        A.Affine(
            shear={'x': (-7, 7), 'y': (-7, 7)},  # จากเดิม ±10
            scale=(0.90, 1.05),
            border_mode=cv2.BORDER_CONSTANT,
            fill=bg_color_rgb,
            p=0.60
        )
    ])

    # -------------------------------------------------
    # 4) AUGMENT LOOP
    # -------------------------------------------------
    generated = 0

    for filename, quota in zip(original_files, quota_list):
        if quota == 0:
            continue

        img_path  = os.path.join(input_dir, filename)
        image_bgr = cv2.imread(img_path)

        if image_bgr is None:
            print(f"⚠️ Cannot read {filename}, skipped.")
            continue

        image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

        for _ in range(quota):
            aug_result = augment_pipeline(image=image)
            aug_img    = aug_result["image"]

            base, ext = os.path.splitext(filename)
            out_name  = f"{base}_aug_{generated}{ext}"

            cv2.imwrite(
                os.path.join(output_dir, out_name),
                cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR)
            )
            generated += 1

    if generated > 0:
        print(f"✔ Generated augmented images: {generated}")

    # -------------------------------------------------
    # 5) COPY ORIGINAL IMAGES
    # -------------------------------------------------
    print("⏳ Copying original images...")
    for filename in original_files:
        src = os.path.join(input_dir, filename)
        dst = os.path.join(output_dir, filename)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)

    final_count = len([
        f for f in os.listdir(output_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))
    ])

    print(f"✅ DONE — Final images in folder: {final_count}")

# =====================================================
# RUN
# =====================================================
if __name__ == "__main__":
    augment_then_copy_originals(
        input_dir=INPUT_IMAGE_DIR,
        output_dir=OUTPUT_IMAGE_DIR,
        max_reference=MAX_CLASS_REFERENCE,
        bg_color_rgb=BG_COLOR_RGB
    )

Class               : Eosinophil
Original images     : 59
Target Reference    : 1150
Base augments/image : 18 times
Images getting +1   : 29 images
Exact total to gen  : 1091 images
✔ Generated augmented images: 1091
⏳ Copying original images...
✅ DONE — Final images in folder: 1150
